# Numerical differentiation

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain the difference between analytical and numerical differentiation
2. implement forward, backward and central differences
3. investigate how step size and round-off error affect the result
4. differentiate experimental data and interpret what the numerical derivative means chemically
```

## The derivative

You may know the derivative as $f'(x)$. Another common notation is

$$\frac{df}{dx}.$$

This is **Leibniz notation** and means “the change in $f$ with respect to $x$”. In chemistry this notation is particularly useful because the variables often have a physical meaning:

$$\frac{dc}{dt}$$

is the change in concentration with time, whereas

$$\frac{d\mathrm{pH}}{dV}$$

is the change in pH with added volume.

The two notations describe the same quantity when $f$ is a function of $x$:

$$f'(x)=\frac{df}{dx}.$$

The derivative is defined as a limit:

$$f'(x)=\frac{df}{dx}=\lim_{\Delta x\rightarrow 0}\frac{f(x+\Delta x)-f(x)}{\Delta x}.$$

On a computer we cannot use an infinitely small $\Delta x$. Instead, we choose a small but finite step $h$:

$$f'(x)\approx\frac{f(x+h)-f(x)}{h}.$$

This is called the **forward difference**.

```{admonition} Exercise along the way
:class: tip
Calculate $f'(1)$ numerically for $f(x)=2x+2$ using $h=10^{-8}$. What do you expect from analytical differentiation?
```

In [ ]:
def f(x):
    return 2*x + 2

x = 1.0
h = 1e-8

f_derivative = (f(x + h) - f(x)) / h
print("Numerical:", f_derivative)
print("Analytical:", 2.0)

We can package the forward difference in a function:

In [ ]:
def derivative_forward(f, x, h=1e-8):
    return (f(x + h) - f(x)) / h

## Error analysis: a smaller step is not always better

It is tempting to think that $h$ should be as small as possible. However, two error mechanisms compete:

- **Approximation error** generally decreases as $h$ becomes smaller.
- **Floating-point round-off error** can become important when we subtract two nearly equal numbers and divide by a very small number.

There is therefore no single universally optimal value of $h$. It depends on the function, the numerical scale and the finite-difference method.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return 2*x**2 + x - 5

def f_derivative_analytical(x):
    return 4*x + 1

x0 = 1.0
h_values = np.logspace(-1, -16, 16)
exact = f_derivative_analytical(x0)

errors = []
for h in h_values:
    numerical = derivative_forward(f, x0, h)
    errors.append(abs(numerical - exact))

plt.loglog(h_values, errors, "o-")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.show()

## Forward, backward and central differences

The forward difference uses the point in front of $x$:

$$\frac{df}{dx}\approx\frac{f(x+h)-f(x)}{h}.$$

The backward difference uses the point behind $x$:

$$\frac{df}{dx}\approx\frac{f(x)-f(x-h)}{h}.$$

The central difference uses points on both sides:

$$\frac{df}{dx}\approx\frac{f(x+h)-f(x-h)}{2h}.$$

For the same step size, the central difference normally has a smaller approximation error than the two one-sided differences.

In [ ]:
def derivative_backward(f, x, h=1e-5):
    return (f(x) - f(x - h)) / h

def derivative_central(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2*h)

def f(x):
    return np.sin(x)

x0 = 1.0
exact = np.cos(x0)

for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    forward = derivative_forward(f, x0, h)
    backward = derivative_backward(f, x0, h)
    central = derivative_central(f, x0, h)
    print(
        f"h={h:g}: forward={forward:.8f}, backward={backward:.8f}, "
        f"central={central:.8f}, exact={exact:.8f}"
    )

### Try it yourself

Compare the three difference methods for different step sizes.

<iframe src="../../basthon/?from=examples/numerical_differentiation_methods.py" width="100%" height="620" frameborder="0" title="Try it yourself: numerical differentiation" loading="lazy" allowfullscreen></iframe>

## Numerical differentiation of experimental data

For experimental data, we do not have a function $f(x)$ that we can evaluate wherever we want. The measurement points determine the distances between the $x$ values.

This makes numerical differentiation very useful, but also more vulnerable: **differentiation often amplifies measurement noise**, because small differences between neighbouring points are divided by a small interval.

We use a titration dataset as an example. The measured volumes are not perfectly evenly spaced, which is common for real experimental data.

In [ ]:
import pandas as pd

data = pd.read_csv("data/titration_data.csv")
data.head()

In [ ]:
volume = data["volume"].to_numpy()
pH = data["pH"].to_numpy()

plt.plot(volume, pH, "o-")
plt.xlabel("Added volume (mL)")
plt.ylabel("pH")
plt.show()

### Differences between neighbouring measurements

For two neighbouring measurements,

$$\frac{\Delta y}{\Delta x}
=\frac{y_{i+1}-y_i}{x_{i+1}-x_i},$$

is the **average slope over the interval** from $x_i$ to $x_{i+1}$. The most natural place to assign this value is therefore the middle of the interval:

$$x_{\mathrm{mid}}=\frac{x_i+x_{i+1}}{2}.$$

Note that this is the **average of the two $x$ values**, not half the difference $(x_{i+1}-x_i)/2$.

If we instead place the difference at $x_i$, the numerical derivative is shifted to the left relative to the interval it actually describes.

In [ ]:
dpH = np.diff(pH)
dV = np.diff(volume)
slope = dpH / dV
volume_mid = (volume[:-1] + volume[1:]) / 2

i_max = np.argmax(slope)
print(f"The largest interval slope occurs around {volume_mid[i_max]:.2f} mL.")

plt.plot(volume_mid, slope, "o-")
plt.xlabel("Added volume, interval midpoint (mL)")
plt.ylabel(r"$\Delta\mathrm{pH}/\Delta V$ (mL$^{-1}$)")
plt.show()

For these data, the largest slope lies between 33.52 and 33.56 mL. The midpoint is therefore **33.54 mL**. This is a better estimate of the position of that interval slope than using 33.52 mL directly.

This does not mean that the true equivalence point is known to one hundredth of a millilitre. Measurement frequency, measurement uncertainty and the chosen differentiation method limit the precision.

### `np.gradient`: a practical method

Once we understand the finite differences ourselves, NumPy can do the calculation for us. `np.gradient(y, x)` uses central differences at interior points and one-sided differences at the endpoints. It can also handle unevenly spaced $x$ values, as in the titration data.

In [ ]:
gradient = np.gradient(pH, volume)
i_max = np.argmax(gradient)

print(f"The maximum from np.gradient occurs at {volume[i_max]:.2f} mL.")

plt.plot(volume, gradient, "o-")
plt.xlabel("Added volume (mL)")
plt.ylabel(r"$d\mathrm{pH}/dV$ (mL$^{-1}$)")
plt.show()

## Short summary

- Numerical differentiation approximates a derivative from finite differences.
- Forward and backward differences use points on one side; the central difference uses points on both sides.
- A smaller step reduces approximation error only until floating-point round-off becomes important.
- For interval differences in experimental data, the derivative belongs naturally at the interval midpoint.
- `np.gradient` is a practical tool once we understand the principle.
- Differentiation can amplify experimental noise.

## Exercises

```{admonition} Exercise 1 – numerical and analytical
:class: tip
Calculate $f'(1)$ numerically and check by analytical differentiation for:

1. $f(x)=x^2-4x+5$
2. $f(x)=e^x$
3. $f(x)=\sqrt{\ln x}$
```

```{admonition} Exercise 2 – error as a function of step size
:class: tip
Compare the forward and central differences for $f(x)=\sin x$ at $x=1$. Make a log-log plot of the absolute error for $h$ from $10^{-1}$ to $10^{-15}$. Comment on the shapes of the curves.
```

```{admonition} Exercise 3 – reaction rate from concentration data
:class: tip
You have measured the concentration of A in a reaction:

`t = [0, 10, 20, 30, 40, 50]` s

`c = [1.00, 0.82, 0.68, 0.56, 0.47, 0.40]` mol/L

Calculate $\Delta c/\Delta t$ between each pair of measurements and plot the reaction rate $-\Delta c/\Delta t$ against the midpoints of the time intervals.
```

```{admonition} Exercise 4 – titration
:class: tip
Use the titration data from this chapter.

1. Find the equivalence point with `np.diff`.
2. Explain why the $x$ values of the derivative should be $(x_i+x_{i+1})/2$.
3. Find the maximum with `np.gradient`.
4. Compare the estimates and comment on how many digits it is reasonable to report.
```

```{admonition} Exercise 5 – potential energy
:class: tip
A simplified potential energy is $U(r)=r^{-12}-2r^{-6}$. Use the central difference to find approximately where $dU/dr=0$. Check the result by plotting $U(r)$.
```

```{admonition} Exercise 6 – unevenly spaced data
:class: tip
Create a small dataset with unevenly spaced time points and a known function. Compare a calculation that incorrectly assumes a constant $\Delta t$ with `np.gradient(y, t)`.
```

```{admonition} Exercise 7 – measurement noise
:class: tip
Create synthetic data from $c(t)=e^{-0.1t}$ and add a small amount of random noise. Differentiate both the noise-free and noisy data numerically. What happens to the noise?
```

```{admonition} Exercise 8 – choose a representation
:class: tip
Explain in your own words the difference between $f'(x)$, $df/dx$, $\Delta y/\Delta x$, and a numerical approximation to $df/dx$.
```